**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/du-2026"], check=True)
    os.chdir("/content/encore/entries/du-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Du, Lai, Hu & Ye (2026): matched-seed synthetic calibration

Paper: doi:10.64898/2026.08.04.742876 (bioRxiv preprint posted 2026-08-10; claims file written from
its full text without the supplement). These claims are `data: elsewhere`: the data are not
deposited but generated by the paper's code (github.com/gwkuqgfkqe/neural-decoder-context-audit, tag
v1.0-jne-submission, commit b042fbe, `synthetic.py` and `run_synthetic_calibration.py`), ported to
`du2026.generate_synthetic` and run here. No dataset version applies; the generator's code is pinned
at that commit. The full claims table is in `02_link_development.ipynb`.

| Claim | Status |
|---|---|
| synthetic-template-null | tested |
| synthetic-signal-ordering | tested |
| synthetic-pair-breaking | tested |

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())
T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)

# Methods 4.8: 480 trials, 60 bins, 8 conditions, 48 neural features, AR(1) 0.85, template scale 1.0,
# neural noise SD 0.20; regimes (trial-signal scale, embedding) 0.20/0, 0.25/0.8, 0.35/1.8; seeds 20260726-35.
# Seed sensitivity (protocol, author step 3): the ten seeds are also run shifted by 1000, 2000, 3000, 4000.
seed_sets = {k: [s + k for s in d.SYNTHETIC_SEEDS] for k in (0,) + EXTRA}
args = [(s, regime) for k in seed_sets for s in seed_sets[k] for regime in d.SYNTHETIC_REGIMES]
t = time.time()
rows = d.run_parallel(d.synthetic_job, args)
syn = pd.DataFrame(rows)
syn["offset"] = [(a[0] - 20260726) // 1000 * 1000 for a in args]
print(f"{len(rows)} fits in {(time.time() - t) / 60:.1f} min")

In [ ]:
ds = d.generate_synthetic(seed=20260726, **d.SYNTHETIC_REGIMES["strong"])
print("generated arrays:", "neural", ds.neural.shape, "context", ds.context.shape, "behavior", ds.behavior.shape)
print("conditions:", np.bincount(ds.condition[:, 0]), "; train/test trials:", len(ds.train_indices), len(ds.test_indices))
print(f"behavior range {ds.behavior.min():.3f} to {ds.behavior.max():.3f}; neural range {ds.neural.min():.3f} to {ds.neural.max():.3f}")
main = syn[syn.offset == 0]
print(main.pivot(index="seed", columns="regime", values="gain").round(5).to_string())
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## synthetic-template-null

> In the matched-seed synthetic calibration, the template-only regime (no trial signal in the neural
> features) gave a median context-conditional neural gain of about zero (-0.0000023, range -0.0000588
> to 0.0000248 over ten seeds).

In [ ]:
for k in (0,) + EXTRA:
    g = syn[(syn.offset == k) & (syn.regime == "template_only")].gain
    print(f"seeds 20260726+{k}..: template-only median {g.median():.7f}, median |gain| {g.abs().median():.7f}, range {g.min():.7f} to {g.max():.7f}")
print("criterion: median absolute gain below 0.03 (expected median -0.0000023, range -0.0000588 to 0.0000248)")

**Result: reproduced.** Over the ten seeds the template-only median gain is -0.0000023 (median absolute gain 0.0000092, far below 0.03), with range -0.0000588 to 0.0000248, the paper's values. The four shifted seed sets give medians of -0.0000040 to -0.0000168.

## synthetic-signal-ordering

> Weak and strong trial-signal regimes gave median gains of 0.2245 and 0.3736, and the three regime
> medians were strictly ordered (template-only, weak, strong).

In [ ]:
for k in (0,) + EXTRA:
    med = syn[syn.offset == k].groupby("regime").gain.median()
    print(f"seeds +{k}: medians template {med['template_only']:.5f}, weak {med['weak']:.5f}, strong {med['strong']:.5f}; "
          f"strictly increasing: {med['template_only'] < med['weak'] < med['strong']}")
print("expected weak 0.2245 +- 0.02245, strong 0.3736 +- 0.03736")
for regime in ("weak", "strong"):
    g = main[main.regime == regime].gain
    print(f"{regime}: range over the ten seeds {g.min():.4f} to {g.max():.4f}")

**Result: reproduced.** The weak and strong medians are 0.22451 and 0.37360 (paper 0.2245 and 0.3736), and the three medians increase strictly; the four shifted seed sets give 0.21996 to 0.22321 and 0.37035 to 0.37368, also ordered.

## synthetic-pair-breaking

> In the strong synthetic regime, correct-pair gain exceeded nearest source-context replacement and the
> mean of 100 within-condition permutations for 10 of 10 seeds; replacement and permutation means were
> -0.3486 and -0.3663.

In [ ]:
for k in (0,) + EXTRA:
    s_ = syn[(syn.offset == k) & (syn.regime == "strong")]
    above = int(((s_.gain > s_.replacement_gain) & (s_.gain > s_.permutation_mean)).sum())
    print(f"seeds +{k}: seeds with gain above both controls {above}/10; mean replacement {s_.replacement_gain.mean():.4f}; "
          f"mean permutation {s_.permutation_mean.mean():.4f}")
print("expected 10 (tolerance 1), -0.3486 +- 0.03486, -0.3663 +- 0.03663")
s_ = main[main.regime == "strong"]
print(s_[["seed", "gain", "replacement_gain", "permutation_mean"]].round(4).to_string())
for regime in ("template_only", "weak"):
    r_ = main[main.regime == regime]
    print(f"{regime}: mean replacement {r_.replacement_gain.mean():.4f}, mean permutation {r_.permutation_mean.mean():.4f}")
print(f"run time {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** In the strong regime the correct-pair gain exceeds both nearest replacement and the permutation mean for 10 of 10 seeds; the mean replacement gain is -0.3486 and the mean permutation gain -0.3663, the paper's values. The shifted seed sets give 10 of 10 each, replacement means -0.3514 to -0.3782 and permutation means -0.3619 to -0.3673, all within tolerance.

## Summary

| Claim | Outcome |
|---|---|
| synthetic-template-null | reproduced |
| synthetic-signal-ordering | reproduced |
| synthetic-pair-breaking | reproduced |

The generator and estimator are ports of the paper's code; the values equal the paper's at the
printed precision. These claims test the code, not a dataset (`data: elsewhere`).